In [ ]:
import emcee
import time
import numpy as np
import sys
import pandas as pd
import os
import subprocess as sp
import struct
from math import log, exp, sqrt, cos, sin, pi
import matplotlib.pyplot as plt

if '../../SelfCalGroupFinder/py/' not in sys.path:
    sys.path.append('../../SelfCalGroupFinder/py/')
if 'SelfCalGroupFinder/py/' not in sys.path:
    sys.path.append('SelfCalGroupFinder/py/')
if 'latsham/py/' not in sys.path:
    sys.path.append('latsham/py/')
from caldata import CalData
from latsham_utils import *
from dataloc import PARAMS_BGSY1_FOLDER
from footprintmanager import FootprintManager
from pyutils import DEGREES_ON_SPHERE, get_volume_at_z

LATSHAM_BIN = "/mount/sirocco1/imw2293/GROUP_CAT/latsham/build/latsham"

### For peeking at halo properties

In [ ]:
HALO_CATALOG = "/mount/sirocco1/imw2293/GROUP_CAT/DATA/POPMOCK/smdpl_z0.19717.M8E9.C.h5";
# contains ['halos'], find it's columns and read it all in

import h5py
with h5py.File(HALO_CATALOG, 'r') as f:
    halos = f['halos']['table']
    print(halos.dtype.names)
    halos_data = halos[:]


print(len(halos_data))

In [ ]:
halos_data

### Define MCMC methods

In [ ]:
def get_mock_wp(path):
    output_mock = np.loadtxt(path)
    wp_m = output_mock[:,4]
    return wp_m

caldata =  CalData.BGS_Y3_Test1(19.5)
IDX_START = 3 # Let's skip first 3 bins due to noise right now

def calc_wp_from_mocks():
    t1 = time.time()
    
    # Define the path to the Corrfunc binary
    corrfunc_path = "/export/sirocco1/tinker/src/Corrfunc/bin/"
    if not os.path.exists(os.path.join(corrfunc_path, "wp")):
        corrfunc_path = "/mount/sirocco1/tinker/src/Corrfunc/bin/"

    output_folder = "/mount/sirocco1/imw2293/GROUP_CAT/OUTPUT/LATSHAM/"
    nthreads = os.cpu_count()
    pimax = 40
    boxsize = 400 # Mpc/h

    # Force a sync to ensure the mock files from the C++ process are fully written to disk
    # before we try to read them with corrfunc. This prevents a potential race condition.
    sp.run(["sync"], check=True)

    if not os.path.exists(f"{corrfunc_path}/wp"):
        print(f"Corrfunc wp executable not found at {corrfunc_path}/wp. Skipping wp calculation.")
        return

    # Call corrfunc compiled executable to compute wp on the mocks from latsham
    for i in range(len(caldata.magbins) - 1):
        for j in range(len(caldata.gmrbins[i]) - 1):
            mag_min = caldata.magbins[i]
            mag_max = caldata.magbins[i + 1]
            gmr_min = caldata.gmrbins[i][j]
            gmr_max = caldata.gmrbins[i][j + 1]
            mock_fname = f"mock_{mag_min:.4f}_{mag_max:.4f}_{gmr_min:.4f}_{gmr_max:.4f}.dat"
            wp_output_fname = f"wp_mock_{mag_min:.4f}_{mag_max:.4f}_{gmr_min:.4f}_{gmr_max:.4f}.dat"
            mock_path = os.path.join(output_folder, mock_fname)
            wp_output_path = os.path.join(output_folder, wp_output_fname)

            cmd = f"{corrfunc_path}/wp {boxsize} {mock_path} a {caldata.rpbinsfile} {pimax} {nthreads} > {wp_output_path} 2> wp_stderr.txt"
            result = sp.run(cmd, cwd=output_folder, shell=True, check=True)
            if result.returncode != 0:
                print(f"Error running corrfunc!")

    t2 = time.time()

    print(f"Calculated all wp in {t2 - t1:.2f} seconds.")

def get_mock_err(volume, wp_err, wp_m):
    vfac = (volume / 400.0**3)**.5 # factor by which to multiply errors
    efac = 0.1 # TODO tweak eventually
    wp_m_err = vfac*wp_err + efac*wp_m
    return wp_m_err

def chisqr():
    chivec = []
    for i in range(len(caldata.magbins) - 1):
        for j in range(len(caldata.gmrbins[i]) - 1):
            mag_min = caldata.magbins[i]
            mag_max = caldata.magbins[i + 1]
            gmr_min = caldata.gmrbins[i][j]
            gmr_max = caldata.gmrbins[i][j + 1]
                    
            # Read in the wp data from the data
            wp, wp_err, r = caldata.get_wp_new(i, j)
            r = r[IDX_START:]
            wp = wp[IDX_START:]
            wp_err = wp_err[IDX_START:]

            # mock
            wp_m = get_mock_wp(f"/mount/sirocco1/imw2293/GROUP_CAT/OUTPUT/LATSHAM/wp_mock_{mag_min:.4f}_{mag_max:.4f}_{gmr_min:.4f}_{gmr_max:.4f}.dat")
            wp_m = wp_m[IDX_START:]
            # If anything in nan, immediatly make these params very bad chisqr
            if np.isnan(wp_m).any():
                print(f"NaN found in wp for magbin {mag_min}-{mag_max}, color {gmr_min}-{gmr_max}. Assigning infinite chi^2.")
                return np.inf
            wp_m_err = get_mock_err(caldata.measurement_volumes[i], wp_err, wp_m)

            sigma = wp_err**2 + wp_m_err**2
            chivec.append(np.sum(((wp - wp_m) / sigma)**2))

    chi2 = np.sum(chivec)
    print(f"Chi^2: {chi2:.2f}")
    return chi2

In [ ]:
# Must keep this protocol syncronized with the C++ code in groups.hpp
MSG_REQUEST = 0
MSG_COMPLETED = 6
MSG_ABORTED = 7
TYPE_FLOAT = 0
TYPE_DOUBLE = 1

def lnlike(params):
    
    # Send parameters to latsham
    msg = struct.pack("<BBI", MSG_REQUEST, TYPE_DOUBLE, len(params)) + struct.pack(f"<{len(params)}d", *params)
    proc.stdin.write(msg)
    proc.stdin.flush()

    while proc.poll() is None: 
        header = pipe_reader.read(6)
        if len(header) == 0:
            continue
        if len(header) < 6:
            raise Exception("Incomplete header, len=" + str(len(header)))
        
        msg_type, data_type, count = struct.unpack("<BBI", header)

        if msg_type not in (MSG_COMPLETED, MSG_ABORTED):
            raise Exception("Unexpected response")
        if data_type not in (TYPE_FLOAT, TYPE_DOUBLE):
            raise Exception("Unexpected data type")
        dtype_marker = 'd' if data_type == TYPE_DOUBLE else 'f'
        dtype = np.dtype(dtype_marker)
        bytes_needed = count * (8 if data_type == TYPE_DOUBLE else 4)

        payload = pipe_reader.read(bytes_needed)
        if len(payload) < bytes_needed:
            raise Exception(f"Incomplete payload, expected {bytes_needed} bytes but got {len(payload)} bytes") 
        data = struct.unpack(f"<{count}{dtype_marker}", payload)
        assert count == len(data)

        if msg_type == MSG_COMPLETED:
            #print("Latent SHAM completed successfully.")
            calc_wp_from_mocks()
            return -0.5 * chisqr()
            
        elif msg_type == MSG_ABORTED:
            print ("Latent SHAM aborted. Returning -inf lnlike.")
            return -np.inf
        
        else:
            raise Exception(f"Unexpected message type: {msg_type}")

    raise Exception(f"Group Finder process ended. What happened?")

# Step on the surface of a (n-1)-hypersphere (unit radius). 
# For the full 4 halo parameters to a galaxy property, use:
# w1 = cos(theta1)
# w2 = sin(theta1)cos(theta2)
# w3 = sin(theta1)sin(theta2)cos(theta3)
# w4 = sin(theta1)sin(theta2)sin(theta3)
def lnprior(params):
    if len(params) == 1:
        scatter = params[0]
        if scatter < 0.00001 or scatter > 10.0:
            return -np.inf
    if len(params) == 2:
        raise NotImplementedError("2D parameter space not implemented yet.")
    if len(params) == 6:
        t1a, t1b, t1c, t2a, t2b, t2c = params

        if not (0 <= t1a <= np.pi):
            return -np.inf
        if not (0 <= t1b <= np.pi):
            return -np.inf
        if not (0 <= t1c < 2*np.pi):
            return -np.inf

        if not (0 <= t2a <= np.pi):
            return -np.inf
        if not (0 <= t2b <= np.pi):
            return -np.inf
        if not (0 <= t2c < 2*np.pi):
            return -np.inf

        return (
            2*np.log(np.sin(t1a))
            + np.log(np.sin(t1b))
            + 2*np.log(np.sin(t2a))
            + np.log(np.sin(t2b))
        )
    return 0.0

def lnprob(theta_params):
    lp = lnprior(theta_params)
    if lp == -np.inf:
        return -np.inf
    
    # The 2gal prop from 2halo prop model
    w_params = spherical_to_linear_parameters(theta_params)
    
    return lp + lnlike(w_params)

# Run

In [ ]:
read_fd, write_fd = os.pipe()
proc = sp.Popen([LATSHAM_BIN, "-P", str(write_fd)], stdin=sp.PIPE, pass_fds=(write_fd,))
os.close(write_fd)
pipe_reader = os.fdopen(read_fd, "rb")

In [ ]:
#backfile = "latsham_mcmc_L2p2p_test4.h5"
backfile = "latsham_mcmc_L2p4p_test2.h5"
backend = emcee.backends.HDFBackend(backfile)
nwalkers = 14
ndim = 6
sampler = emcee.EnsembleSampler(nwalkers, ndim, lnprob, backend=backend)
# Want to start walkers near 0.0 for theta1 and pi/2 for theta2 to start near the "identity" transform in the latent space, which is a reasonable place to start for testing and debugging. We can explore more of the parameter space once we verify that the MCMC is working correctly and that the lnlike calculations are producing sensible results.
#init_pos = [np.array([0.0, pi / 2.0]) + 0.1 * np.random.randn(ndim) for i in range(nwalkers)]
start = [0.999, -0.137, 0.26, -0.008, -0.111, 0.422, 1.183, -0.168]
init_pos = [np.array(linear_to_spherical_parameters(start * (1 + 0.2*np.random.randn(8)))) for i in range(nwalkers)]

#init_pos = 1.0 * np.random.uniform(0.0, 2 * pi, (nwalkers, ndim))

In [ ]:
# RUN MCMC
#sampler.run_mcmc(None, 100, progress=True)
sampler.run_mcmc(init_pos, 100, progress=True)

In [ ]:
# Kill latsham if needed
if proc.poll() is None:
    print("Process is still running, killing now.")
    # Kill it
    proc.terminate()
else:
    print(f"Process finished with code {proc.returncode}")
    # Note -15 is SIGTERM which is what happens when you kill, which is terminate() above

In [ ]:
chains = sampler.get_chain().reshape(-1, ndim)
logprobs = sampler.get_log_prob().reshape(-1)

import corner
fig = corner.corner(chains.reshape(-1, ndim))

In [ ]:
# Show best parameter and chisqr
best_idx = np.argmax(logprobs)
best_params = chains.reshape(-1, ndim)[best_idx]
print(f"Best fit params: {best_params}")
linp = spherical_to_linear_parameters(best_params)
print(f"Linear Space best fit params: {linp}")
print(f"Best fit chi^2: {-2*logprobs[best_idx]}")

In [ ]:
# Run latsham with best params, no pipe. use linp
proc = sp.Popen([LATSHAM_BIN, "-m", f"{linp[0]},{linp[1]},{linp[2]},{linp[3]},{linp[4]},{linp[5]},{linp[6]},{linp[7]}"],)

# Results

In [ ]:
# Will use outputs from whatever is last run of latsham! So run for best parameters

def gmr_to_color_absolute(gmr):
    cmin = 0.3
    cmax = 1.0
    cmap = plt.get_cmap("seismic")   # or "RdBu", "turbo", "viridis"
    normalized = np.clip((gmr-cmin)/(cmax-cmin), 0, 1)
    rgb = cmap(normalized)[:3]
    return rgb

def gmr_to_color(gmr, gmrbins):
    cmin = min(gmrbins)
    cmax = max(gmrbins)
    cmap = plt.get_cmap("seismic")   # or "RdBu", "turbo", "viridis"
    normalized = np.clip((gmr-cmin)/(cmax-cmin), 0, 1)
    rgb = cmap(normalized)[:3]
    return rgb

calc_wp_from_mocks()
chi = chisqr()
print(f"Chi^2: {chi:.2f}")
num = len(caldata.magbins) - 1
nrows = 2
ncols = 5
fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(2 + 3 * ncols, 4 * nrows), dpi=200)
axes = np.array(axes).reshape(-1)  # flatten for easy indexing

for idx in range(len(caldata.magbins) - 1):
    offset = 1.0
    for j in range(len(caldata.gmrbins[idx]) - 1):
        ax = axes[idx]
        a=0.25

        wp, wp_err, radius = caldata.get_wp_new(idx, j)

        color = gmr_to_color((caldata.gmrbins[idx][j] + caldata.gmrbins[idx][j+1]) / 2.0, caldata.gmrbins[idx])
        ax.errorbar(radius, wp*offset, yerr=wp_err*offset, fmt='o', markersize=5, markeredgewidth=1.4, markeredgecolor='k', markerfacecolor=color, capsize=5, elinewidth=1.5, ecolor='k')

        wp_mock = get_mock_wp((f"/mount/sirocco1/imw2293/GROUP_CAT/OUTPUT/LATSHAM/wp_mock_{caldata.magbins[idx]:.4f}_{caldata.magbins[idx+1]:.4f}_{caldata.gmrbins[idx][j]:.4f}_{caldata.gmrbins[idx][j+1]:.4f}.dat"))      
        wp_mock_err = get_mock_err(caldata.measurement_volumes[idx], wp_err, wp_mock)

        ax.plot(radius, wp_mock*offset, '-', color=color, alpha=1.0)
        ax.fill_between(radius, wp_mock*offset - wp_mock_err*offset, wp_mock*offset + wp_mock_err*offset, color=color, alpha=a)

        # Plot config
        ax.set_xscale('log')
        ax.set_yscale('log')
        ax.set_xlabel('$r_p$ [Mpc $h^{-1}$]')
        ax.set_ylabel('$w_p(r_p)$')
        ax.set_ylim(1, 4000)
        ax.set_title(f'[{caldata.magbins[idx]}, {caldata.magbins[idx+1]}]')

        offset *= 2.0  # Increase offset for next color bin


fig.tight_layout()
fig.show()

In [ ]:
# Read in the full mock
path = "/mount/sirocco1/imw2293/GROUP_CAT/OUTPUT/LATSHAM/full_mock.dat"
df_full_mock = pd.read_csv(path, delim_whitespace=True, 
                           names=["x", "y", "z", "logMh", "a", "λ", "c", "Mr", "g-r"])

In [ ]:
# Let's look at the correlation matrix of the full mock
df_features = df_full_mock[["logMh", "a", "λ", "c", "Mr", "g-r"]]
corr = df_features.corr()

import matplotlib.pyplot as plt
import seaborn as sns
plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, cmap="coolwarm", vmin=-1, vmax=1)
plt.title("Correlation Matrix of Full Mock")
plt.show()